# Process Representation 
## Class Process

In [62]:
class Process:
    def __init__(self, pid, arrival_time, burst_time, priority=0):
        self.pid = pid
        self.arrival_time = arrival_time
        self.burst_time = burst_time
        self.priority = priority
        self.remaining_time = burst_time
        self.start_time = None
        self.completion_time = None
        self.waiting_time = 0
        self.turnaround_time = 0

# Scheduling Alogrithms and Performance Metrics

## FIRST COME FIRST SERVE

In [2]:

def fcfs(processes):
    processes.sort(key=lambda p: p.arrival_time)
    # the sorting is only done to jump to next process quickly
    current_time = 0
    schedule = []
    total_burst_time = 0
    avg_turnaround_time,avg_waiting_time,cpu_utilisation = 0,0,0
    for process in processes:
        process.start_time = max(current_time, process.arrival_time)
        # if a process came before the other ends , he waits , else he directly starts running
        current_time = process.start_time + process.burst_time
        process.completion_time = current_time
        process.turnaround_time = process.completion_time - process.arrival_time
        process.waiting_time = process.turnaround_time - process.burst_time
        total_burst_time += process.burst_time
        avg_turnaround_time += process.turnaround_time
        avg_waiting_time += process.waiting_time
        cpu_utilisation += process.burst_time
        
        schedule.append(process.pid)

    makespan = processes[-1].completion_time - processes[0].arrival_time
    cpu_utilisation = ( total_burst_time/ makespan) * 100  # in percentage
    avg_turnaround_time /= len(processes)
    avg_waiting_time /= len(processes)
    return schedule, avg_turnaround_time,   avg_waiting_time, cpu_utilisation

processes = [
    Process(1, 1, 5),
    Process(2, 6, 3),
    Process(3, 2, 8),
    Process(4, 3, 6)
]
fcfs(processes)

([1, 3, 4, 2], 12.75, 7.25, 100.0)

## SHortest JOb FIrst

In [3]:

def sjf(processes): 
    l = len(processes)
    processes.sort(key=lambda x: (x.arrival_time, x.burst_time))
    current_time, schedule = 0, []
    start_time = min(processes,key = lambda p : p.arrival_time).arrival_time
    avg_turnaround_time,avg_waiting_time,total_burst_time = 0,0,0
    while processes:
        available = [p for p in processes if p.arrival_time <= current_time]
        # all processes that can be runned now
        # if no process is available , we go to the time of the start of next process 
        if not available:
            current_time = processes[0].arrival_time
            continue
        process = min(available, key=lambda p: p.burst_time)
        #the process with the shortest burst time is selected
        processes.remove(process)
        process.start_time = current_time
        current_time += process.burst_time
        process.completion_time = current_time
        process.turnaround_time = process.completion_time - process.arrival_time
        process.waiting_time = process.turnaround_time - process.burst_time
        avg_waiting_time += process.waiting_time
        avg_turnaround_time += process.turnaround_time
        total_burst_time += process.burst_time
    
        schedule.append(process.pid)
    avg_turnaround_time /= l
    avg_waiting_time /= l
    makespan = current_time - start_time
    cpu_utilization = (total_burst_time/makespan) *100
    return schedule, avg_turnaround_time, avg_waiting_time,cpu_utilization
processes = [
    Process(1, 0, 5),
    Process(2, 0, 3),
    Process(3, 9, 8),
    Process(4, 10, 6)
]
sjf(processes)



([2, 1, 3, 4], 8.0, 2.5, 95.65217391304348)

## Priority scheduling

In [4]:
def priority_scheduling(processes): # Priority Scheduling (Non-Preemptive)
    # HIGH PRIORITY MEANS LOW NUMBER 
    avg_turnaround_time,avg_waiting_time = 0,0
    l = len(processes)
    processes.sort(key=lambda x: (x.arrival_time, x.priority))
    start_time = min(processes,key = lambda p : p.arrival_time).arrival_time
    current_time, schedule = 0, []
    total_burst_time = 0
    while processes:
        available = [p for p in processes if p.arrival_time <= current_time]
        if not available:
            current_time = processes[0].arrival_time
            continue
        process = min(available, key=lambda p: p.priority)
        processes.remove(process)
        process.start_time = current_time
        current_time += process.burst_time
        process.completion_time = current_time
        process.turnaround_time = process.completion_time - process.arrival_time
        process.waiting_time = process.turnaround_time - process.burst_time
        avg_waiting_time += process.waiting_time
        avg_turnaround_time += process.turnaround_time
        total_burst_time += process.burst_time
        schedule.append(process.pid)
    avg_turnaround_time /= l
    avg_waiting_time /= l
    makespan = (current_time -  start_time)
    cpu_utilization = (total_burst_time/makespan) * 100
    return schedule, avg_turnaround_time, avg_waiting_time,cpu_utilization


processes = [
    Process(1, 9, 10,3),
    Process(2, 0, 6,1),
    Process(3, 0, 2,4),
    Process(4, 9, 4,5),
    Process(5, 9, 8,1)
]    

print(priority_scheduling(processes.copy()))



([2, 3, 5, 1, 4], 12.4, 6.4, 96.7741935483871)


## Round Robin

In [5]:
def round_robin(processes, quantum = 0.01):# Round Robin Scheduling (Preemptive)
    for p in processes:
        p.remaining_time = p.burst_time
    processes.sort(key = lambda p: p.arrival_time )
    queue = []
    current_time =0
    schedule=[]
    remaining = processes[:] #copy
    avg_turnaround_time,avg_waiting_time,total_burst_time = 0,0,0
    start_time = min(processes,key = lambda p : p.arrival_time).arrival_time
    while queue or remaining:
        if queue:
            p = queue.pop(0)
            time_elapsed = min(p.remaining_time,quantum)
            current_time += time_elapsed
            p.remaining_time -= time_elapsed

            # this is just to make sure the ones that came during quantum 
            # will be before the one we just did at next cycle
            while remaining and remaining[0].arrival_time <= current_time :
                queue.append(remaining[0])
                remaining.pop(0)

            if p.remaining_time == 0 :
                p.completion_time = current_time
                p.turnaround_time = p.completion_time - p.arrival_time
                p.waiting_time = p.turnaround_time - p.burst_time
                avg_waiting_time += p.waiting_time
                avg_turnaround_time += p.turnaround_time
                schedule.append(p.pid)
                total_burst_time += p.burst_time
            else:
                queue.append(p)
        else :
            current_time = remaining[0].arrival_time
            while remaining and remaining[0].arrival_time <= current_time :
                queue.append(remaining[0])
                remaining.pop(0)
    avg_turnaround_time /= len(processes)
    avg_waiting_time /= len(processes)
    makespan = current_time - start_time
    cpu_utilisation = (total_burst_time/makespan) * 100


    return schedule, avg_turnaround_time, avg_waiting_time, cpu_utilisation


processes = [
    Process(1, 0, 10,3),
    Process(2, 0, 6,1),
    Process(3, 0, 2,4),
    Process(4, 3, 4,5),
    Process(5, 3, 2,1)
]    

print(round_robin(processes.copy(),1))


([3, 5, 4, 2, 1], 15.2, 10.4, 100.0)


## Priority Round Robin

In [18]:
def priority_rr(processes,quantum):
    processes.sort(key=lambda p: p.arrival_time)
    queue, schedule, current_time = processes[:], [], processes[0].arrival_time
    queue = [p for p in queue if p.arrival_time <= current_time]
    queue = [p for p in queue if p.priority == min([p.priority for p in queue])]
    avg_turnaround_time,avg_waiting_time,total_burst_time = 0,0,0
    while queue:
        process = queue.pop(0)

        if process.arrival_time > current_time:
            current_time = process.arrival_time
        execution_time = min(quantum, process.remaining_time)
        process.remaining_time -= execution_time
        current_time += execution_time
        if process.remaining_time == 0:
            process.completion_time = current_time
            process.turnaround_time = process.completion_time - process.arrival_time
            process.waiting_time = process.turnaround_time - process.burst_time
            avg_waiting_time += process.waiting_time
            avg_turnaround_time += process.turnaround_time
            total_burst_time += process.burst_time
            schedule.append(process.pid)
        else:
            queue.append(process)
        l = [p for p in processes if p.arrival_time <= current_time and p not in queue and p.remaining_time > 0]
        queue.extend(l)
        if not queue:
            future_arrivals = [p.arrival_time for p in processes if p.remaining_time > 0 and p.arrival_time > current_time]
            if future_arrivals:
                   current_time = min(future_arrivals)
                   queue = [p for p in processes if p.arrival_time <= current_time and p.remaining_time > 0]
            else:
                  break  # No future arrivals and queue is empty => all done

        queue = [p for p in queue if p.priority == min([p.priority for p in queue])]

    avg_turnaround_time /= len(processes)
    avg_waiting_time /= len(processes)
    makespan = current_time - processes[0].arrival_time
    cpu_utilisation = (total_burst_time / makespan) * 100
    return schedule, avg_turnaround_time, avg_waiting_time, cpu_utilisation



processes1 = [
    Process(1, 8, 10,3),
    Process(2, 0, 6,1),
    Process(3, 0, 2,4),
    Process(4, 9, 4,5),
    Process(5, 9, 8,1)
]    

print(priority_rr(processes1,1))


([2, 3, 5, 1, 4], 12.2, 6.2, 100.0)


# Input Data

## Reading from CSV file

In [ ]:
file = "processes.txt"
def read_processes_from_file(filepath):
    processes = []
    with open(filepath, 'r') as file:
        for lineno, line in enumerate(file, start=1):
            line = line.strip()
            if not line:
                continue  # skip empty lines

            parts = line.split(',')
            if not (3 <= len(parts) <= 4):
                raise ValueError(f"[Line {lineno}] Expected 3 or 4 values, got {len(parts)}: {line}")

            try:
                parts = list(map(int, parts))
            except ValueError:
                raise ValueError(f"[Line {lineno}] Non-integer value found: {line}")

            pid, arrival, burst = parts[:3]
            priority = parts[3] if len(parts) == 4 else 0

            # Check for negative values
            if any(val < 0 for val in [pid, arrival, burst, priority]):
                raise ValueError(f"[Line {lineno}] Negative value not allowed: {line}")

            processes.append(Process(pid, arrival, burst, priority))
    return processes

## Generate Random processes

In [63]:
import random

def generate_random_processes(n, arrival_range=(0, 10), burst_range=(1, 10), priority_range=(0, 5)):
    processes = []
    for i in range(1, n + 1):
        arrival_time = random.randint(*arrival_range)
        burst_time = random.randint(*burst_range)
        priority = random.randint(*priority_range)
        processes.append(Process(pid=i, arrival_time=arrival_time, burst_time=burst_time, priority=priority))
    return processes


## Listing Processes

In [60]:
def print_processes(processes):
    print(f"{'PID':<5}{'Arrival':<10}{'Burst':<8}{'Priority':<10}{'Remaining':<10}{'Completion':<12}")
    print("-" * 55)
    for p in processes:
        print(f"{p.pid:<5}{p.arrival_time:<10}{p.burst_time:<8}{p.priority:<10}{p.remaining_time:<10}"
              f"{p.completion_time if p.completion_time is not None else '-':<12}")


In [12]:
processes = [
    Process(pid=1, arrival_time=0, burst_time=5, priority=2),
    Process(pid=2, arrival_time=1, burst_time=3, priority=1),
    Process(pid=3, arrival_time=2, burst_time=8, priority=3),
    Process(pid=4, arrival_time=3, burst_time=6, priority=2),
    Process(pid=5, arrival_time=4, burst_time=2, priority=1),
]

p1 = Process(1, 5, 8,1)
p2 = Process(2, 0, 4,3)
p3 = Process(3, 0, 2,1)
p4 = Process(4, 1, 6,4)

print(priority_rr([p1, p2,p3,p4],2))


([3, 2, 1, 4], 9.0, 4.0)


In [14]:
print(fcfs(processes.copy()))
print(sjf(processes.copy()))
print(priority_scheduling(processes.copy()))
print(round_robin(processes.copy(), 1))
print(priority_rr(processes.copy(), 1))


([1, 2, 3, 4, 5], 19.2, 13.2)
([3, 4, 2, 5, 1], 14.0, 8.0)
([2, 5, 1, 3, 4], 20.0, 14.0)
([3, 4, 2, 5, 1], 21.2, 15.2)
([2, 5, 1, 3, 4], 21.0, 15.0)


In [64]:
pip install streamlit


Note: you may need to restart the kernel to use updated packages.


In [68]:
import streamlit as st

st.title("CPU Scheduling Simulator")

# --- Input method ---
input_method = st.selectbox("Select Input Method", ["Manual", "Random", "From File"])

if input_method == "Manual":
    st.subheader("Add Processes")
    num = st.number_input("Number of Processes", min_value=1, max_value=20, step=1)
    processes = []
    for i in range(num):
        with st.expander(f"Process {i+1}"):
            arrival = st.number_input(f"Arrival Time P{i+1}", 0)
            burst = st.number_input(f"Burst Time P{i+1}", 1)
            priority = st.number_input(f"Priority P{i+1}", 1)
            processes.append(Process(i+1, arrival, burst, priority))
elif input_method == "Random":
    num = st.slider("Number of Random Processes", 3, 20)
    processes = generate_random_processes(num)
    st.success("Random processes generated.")
elif input_method == "From File":
    uploaded = st.file_uploader("Upload File", type=["txt", "csv"])
    if uploaded:
        processes = read_processes_from_file(uploaded)

# --- Show Process Table ---
if processes:
    st.subheader("Process Table")
    print_processes(processes)

# --- Algorithm Selection ---
st.subheader("Choose Scheduling Algorithm")
algo = st.selectbox("Algorithm", ["FCFS", "Priority", "RR", "Priority + RR"])

quantum = 1
if "RR" in algo:
    quantum = st.slider("Quantum", 1, 10)

if st.button("Run Scheduling"):
    if algo == "FCFS":
        result = fcfs(processes)
    elif algo == "Priority":
        result = priority(processes)
    elif algo == "RR":
        result = round_robin(processes, quantum)
    else:
        result = priority_rr(processes, quantum)

    schedule, avg_tat, avg_wt, cpu_util = result

    # --- Gantt Chart ---
    st.subheader("Gantt Chart")

    # --- Metrics ---
    st.metric("Average Turnaround Time", f"{avg_tat:.2f}")
    st.metric("Average Waiting Time", f"{avg_wt:.2f}")
    st.metric("CPU Utilization", f"{cpu_util:.2f}%")

# --- Comparison ---
if st.checkbox("Compare All Algorithms"):
    pass


PID  Arrival   Burst   Priority  Remaining Completion  
-------------------------------------------------------
1    0         1       1         1         -           


In [74]:
python3.exe -m pip install --upgrade pip

SyntaxError: invalid syntax (3136599086.py, line 1)